# Notebook 2: Bronze Layer - Raw Data Ingestion

**Objective**: Ingest data from S3 into the lakehouse Bronze layer.

## What is the Bronze Layer?

The **Bronze Layer** is the first layer of the lakehouse:
- It stores **raw data**, as received from the source
- Adds **technical metadata** (ingestion date, filename, etc.)
- Keeps the original format (PascalCase, original types)
- Is **partitioned** by ingestion date for performance

## Data Flow

```
S3 (CSV) ──► Spark Read ──► Metadata Enrichment ──► Iceberg Bronze Table
```

---
## INITIALIZATION

In [ ]:
# Configure project path
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import Spark modules and configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET

# Create Spark session
spark = get_spark("bronze-ingestion")

# Configure Nessie main branch
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Create namespaces
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")

print("=" * 60)
print("BRONZE LAYER - DATA INGESTION")
print("=" * 60)

## BATCH 1 - FIRST INGESTION

**Scenario**: It's morning, the first sales file arrives.

File: `sales_batch_001.csv`

### 1. Read first batch from S3

In [ ]:
# Import PySpark functions
from pyspark.sql.functions import current_timestamp, current_date, lit

# First batch path
batch1_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_001.csv"

# Read with appropriate CSV options
batch1_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch1_path)
)

batch1_count = batch1_raw.count()
print(f"BATCH 1 read: {batch1_count:,} records")

### Raw data preview

In [ ]:
print("=== Raw data preview ===")
batch1_raw.show(5, truncate=False)

print("\n=== Data schema ===")
batch1_raw.printSchema()

### 2. Enrichment with technical metadata

The Bronze layer adds traceability columns:

| Column | Description |
|---------|-------------|
| `ingestion_date` | Ingestion date (partition) |
| `ingestion_ts` | Exact ingestion timestamp |
| `source_file` | Source filename |
| `source_system` | Source system |
| `batch_id` | Batch identifier |

In [ ]:
# Enrichment with metadata
batch1_bronze = (
    batch1_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_001.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_1"))
)

print("=== Enriched data (with metadata) ===")
batch1_bronze.select("Order ID", "Order Date", "Sales", "ingestion_date", "ingestion_ts", "batch_id").show(5, truncate=False)

### 3. Drop old table (if exists)

In [ ]:
# Cleanup: drop tables if they exist (for a clean demo)
spark.sql("DROP TABLE IF EXISTS nessie.bronze.sales")

print("Existing tables dropped")

### 4. Create Iceberg Bronze Table

**Key points**:
- Format: **Iceberg** (ACID table format)
- Partition: by `ingestion_date` (optimizes date-based queries)
- Catalog: **Nessie** (automatic versioning)

In [ ]:
from pyspark.sql.functions import col

# Create Bronze table with partitioning on ingestion_date
batch1_bronze.writeTo("nessie.bronze.sales").using("iceberg").partitionedBy(col("ingestion_date")).create()

print(f"BATCH 1 -> Bronze table created: {batch1_count:,} records")

### 5. Verify Bronze table

In [ ]:
bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"Records in Bronze: {bronze_count:,}")

In [ ]:
print("=== Bronze table sample ===")
spark.sql("SELECT * FROM nessie.bronze.sales LIMIT 5").show(truncate=False)

### 6. Inspect Iceberg metadata

Iceberg keeps a history of all changes. Each write creates a new **snapshot**.

In [ ]:
print("=== Snapshot history ===")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

## BATCH 2 - SECOND INGESTION (APPEND MODE)

**Scenario**: It's noon, a new sales file arrives.

**Key point**: New data is **appended** to existing data (APPEND mode).

In [ ]:
# Read second batch from S3
batch2_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_002.csv"

batch2_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch2_path)
)

batch2_count = batch2_raw.count()
print(f"BATCH 2 read: {batch2_count:,} records")

In [ ]:
# Enrichment and append to Bronze table (APPEND mode)
batch2_bronze = (
    batch2_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_002.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_2"))
)

# APPEND mode: data is added to existing data
batch2_bronze.writeTo("nessie.bronze.sales").append()

new_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 2 -> Added to Bronze table (APPEND)")
print(f"New Bronze total: {new_bronze_count:,} (+{new_bronze_count - batch1_count})")

## BATCH 3 - THIRD INGESTION

In [ ]:
# Read third batch from S3
batch3_path = f"s3a://{AWS_BUCKET}/raw/batches/sales_batch_003.csv"

batch3_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(batch3_path)
)

batch3_count = batch3_raw.count()
print(f"BATCH 3 read: {batch3_count:,} records")

In [ ]:
# Enrichment and append to Bronze
batch3_bronze = (
    batch3_raw
    .withColumn("ingestion_date", current_date())
    .withColumn("ingestion_ts", current_timestamp())
    .withColumn("source_file", lit("sales_batch_003.csv"))
    .withColumn("source_system", lit("sales_system"))
    .withColumn("batch_id", lit("batch_3"))
)

batch3_bronze.writeTo("nessie.bronze.sales").append()

final_bronze_count = spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales").first()[0]
print(f"BATCH 3 -> Added to Bronze table (APPEND)")
print(f"New Bronze total: {final_bronze_count:,} (+{final_bronze_count - new_bronze_count})")

## BRONZE INGESTION SUMMARY

In [ ]:
print("=" * 60)
print("BRONZE INGESTION SUMMARY")
print("=" * 60 + '\n')

print(f"Total Bronze: {final_bronze_count:,} records")

print("Distribution by batch:")
spark.sql("""
    SELECT batch_id, COUNT(*) as count
    FROM nessie.bronze.sales
    GROUP BY batch_id
    ORDER BY batch_id
""").show()

print("\nSnapshot history:")
spark.sql("""
    SELECT 
        made_current_at,
        snapshot_id,
        parent_id
    FROM nessie.bronze.sales.history
    ORDER BY made_current_at
""").show(truncate=False)

print("Observation: Each batch creates a new snapshot!")
print("We can go back to any point in time.")

In [ ]:
print("""
╔════════════════════════════════════════════════════════════╗
║         BRONZE LAYER - INGESTION COMPLETE                  ║
╠════════════════════════════════════════════════════════════╣
║                                                            ║
║  Table               nessie.bronze.sales                   ║
║  Format              Iceberg (ACID, Time Travel)           ║
║  Partition           ingestion_date                        ║
║  Records             {:>10,}                               ║
║  Snapshots           {:>10} (time travel capacity!)        ║
║                                                            ║
║  Metadata added:                                         ║
║    • ingestion_date, ingestion_ts                          ║
║    • source_file, source_system, batch_id                  ║
║                                                            ║
╚════════════════════════════════════════════════════════════╝
""".format(final_bronze_count, spark.sql("SELECT COUNT(*) FROM nessie.bronze.sales.history").first()[0]))

print("→ Next step: Run '03_silver_transformations.ipynb'")